In [17]:
import wandb
import json
import re

ENTITY = "laura-rebollo-crespo-universitat-polit-cnica-de-catalunya"
PROJECT = "AA1"

OUTPUT_FILE = "wandb_all_runs.json"

api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")

print(f"Found {len(runs)} runs")


def deep(obj):

    if isinstance(obj, dict):
        return {k: deep(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [deep(x) for x in obj]

    try:
        import numpy as np
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        if isinstance(obj, np.ndarray):
            return obj.tolist()
    except:
        pass

    if "SummarySubDict" in str(type(obj)):
        return str(obj)

    return obj


def extract_html(run):

    try:
        for f in run.files():

            if "classification" in f.name.lower() and f.name.endswith(".html"):

                path = f.download(replace=True).name

                with open(path, "r", encoding="utf-8", errors="ignore") as file:
                    html = file.read()

                text = re.sub(r"<[^>]+>", "", html)
                text = re.sub(r"\n\s*\n", "\n", text).strip()

                return text

    except:
        return None

    return None


all_runs = []

for i, run in enumerate(runs):

    print(f"[{i+1}/{len(runs)}] {run.name}")

    config = deep(dict(run.config))
    summary = deep(dict(run.summary))

    model_name = config.get("model_name", "UNKNOWN")

    classif_report = summary.get("classification_report")
    
    is_html_dict = isinstance(classif_report, dict) and classif_report.get("_type") == "html-file"
    is_html_str = isinstance(classif_report, str) and "html-file" in classif_report

    if is_html_dict or is_html_str:
        classif_report = extract_html(run)

    if classif_report is None:
        classif_report = extract_html(run)

    run_data = {
        "run_id": run.id,
        "run_name": run.name,
        "model": model_name,
        "hyperparameters": config,
        "metrics": {
            "accuracy": summary.get("val_accuracy"),
            "f1_macro": summary.get("val_f1_macro"),
            "precision_macro": summary.get("val_precision_macro"),
            "recall_macro": summary.get("val_recall_macro"),
        },
        "classification_report": classif_report
    }

    all_runs.append(run_data)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_runs, f, indent=2, ensure_ascii=False)

print(f"\nDONE → exported {len(all_runs)} runs")


Found 96 runs
[1/96] LinearDiscriminantAnalysis_f1-0.7851
[2/96] QuadraticDiscriminantAnalysis_f1-0.7258
[3/96] SVC_f1-0.8978
[4/96] LogisticRegression_f1-0.8321
[5/96] SupervisedClusteringWrapper_f1-0.4836
[6/96] SupervisedClusteringWrapper_f1-0.2495
[7/96] SupervisedClusteringWrapper_f1-0.6017
[8/96] SupervisedClusteringWrapper_f1-0.5091
[9/96] SupervisedClusteringWrapper_f1-0.5197
[10/96] LinearDiscriminantAnalysis_f1-0.7851
[11/96] SVC_f1-0.8895
[12/96] SVC_f1-0.8795
[13/96] SVC_f1-0.8795
[14/96] SVC_f1-0.8795
[15/96] SVC_f1-0.8795
[16/96] SVC_f1-0.8974
[17/96] ExtraTreesClassifier_f1-0.9154
[18/96] SVC_f1-0.8795
[19/96] LogisticRegression_f1-0.8321
[20/96] SVC_f1-0.9008
[21/96] LinearDiscriminantAnalysis_f1-0.7298
[22/96] SVC_f1-0.7943
[23/96] QuadraticDiscriminantAnalysis_f1-0.4968
[24/96] KNeighborsClassifier_f1-0.5471
[25/96] ExtraTreesClassifier_f1-0.7884
[26/96] SVC_f1-0.7294
[27/96] SVC_f1-0.8683
[28/96] SVC_f1-0.8373
[29/96] SVC_f1-0.8434
[30/96] SVC_f1-0.8434
[31/96] TabPF